# Lab 5 (Optional Advanced): The PQC Migration Scanner
## Applied Quantum Computing — Chapter 5: The Cryptocalypse

**Course:** Applied Quantum Computing for Business Leaders  
**Author:** Dr. Ernesto Lee  
**GitHub:** [liquid-books/applied-quantum-computing](https://github.com/liquid-books/applied-quantum-computing)

---

### What This Notebook Does

The most expensive phase of any PQC migration program is the **inventory** — because most organizations don't know where all their cryptographic assets are.

This notebook builds a **PQC Migration Scanner** that:
1. Walks a directory tree and finds every certificate, key, and cryptographic file
2. Extracts metadata: algorithm, key size, expiration date, subject information
3. Applies the **Mosca Inequality** to classify each asset: GREEN / YELLOW / RED
4. Outputs a **prioritized CSV migration report**

### The Mosca Inequality (Quick Review)

**If (data lifetime) + (migration time remaining) > (time to Q-Day), you are already late.**

- 🟢 **GREEN:** Safe — plenty of time to migrate before exposure
- 🟡 **YELLOW:** Warning — within 2 years of the danger threshold  
- 🔴 **RED:** Critical — already in deficit; migration must begin immediately

## Step 1: Install Dependencies

In [ ]:
# Install required libraries
!pip install cryptography --quiet

print("✅ Dependencies installed")

## Step 2: Import Libraries and Configure Parameters

In [ ]:
import os
import csv
import json
import datetime
import tempfile
from pathlib import Path
from typing import Optional, Dict, List, Tuple

# Cryptography library imports
from cryptography import x509
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa, ec, dsa, ed25519, ed448
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric.rsa import RSAPrivateKey, RSAPublicKey
from cryptography.hazmat.primitives.asymmetric.ec import EllipticCurvePrivateKey, EllipticCurvePublicKey
from cryptography.hazmat.primitives.asymmetric.dsa import DSAPrivateKey, DSAPublicKey

# ============================================================
# CONFIGURATION — Adjust these for your organization
# ============================================================

# Q-Day estimate (years from now)
# Conservative: 11 years (2035), Aggressive: 6 years (2030)
QDAY_YEARS_FROM_NOW = 9  # ~2033, midpoint estimate

# Your organization's estimated migration time (years)
# Small org: 2-3 years, Medium: 4-6 years, Large enterprise: 6-10 years
ORG_MIGRATION_YEARS = 4

# Default data lifetime for assets without expiration (years)
DEFAULT_LIFETIME_YEARS = 10

# Yellow zone threshold: assets within this many years of being RED
YELLOW_BUFFER_YEARS = 2

TODAY = datetime.date.today()

print(f"⚙️  Configuration loaded")
print(f"   Q-Day estimate: {TODAY.year + QDAY_YEARS_FROM_NOW}")
print(f"   Organization migration time: {ORG_MIGRATION_YEARS} years")
print(f"   Scan date: {TODAY}")

## Step 3: The Mosca Inequality Engine

In [ ]:
def mosca_classification(
    data_lifetime_years: float,
    migration_years_remaining: float,
    qday_years_from_now: float
) -> Tuple[str, float, str]:
    """
    Apply the Mosca Inequality to classify a cryptographic asset.
    
    The Mosca Inequality: if Z + Y > X, you are already late.
      Z = data lifetime (years)
      Y = migration time remaining (years)
      X = time to Q-Day (years)
    
    Returns: (status, buffer_years, explanation)
      status: 'GREEN', 'YELLOW', or 'RED'
      buffer_years: positive = surplus, negative = deficit
      explanation: human-readable risk description
    """
    total_exposure = data_lifetime_years + migration_years_remaining
    buffer = qday_years_from_now - total_exposure
    
    if buffer > YELLOW_BUFFER_YEARS:
        status = 'GREEN'
        explanation = f"{buffer:.1f} year buffer before Mosca threshold"
    elif buffer > 0:
        status = 'YELLOW'
        explanation = f"Only {buffer:.1f} year(s) of buffer — migration urgency HIGH"
    else:
        status = 'RED'
        explanation = f"DEFICIT of {abs(buffer):.1f} year(s) — already past Mosca threshold"
    
    return status, buffer, explanation


def get_priority(status: str, key_size: Optional[int], algorithm: str) -> str:
    """
    Determine migration priority based on Mosca status and cryptographic strength.
    """
    if status == 'RED':
        # Weak keys in RED are CRITICAL
        if algorithm in ['RSA', 'DSA'] and key_size and key_size < 2048:
            return 'CRITICAL'
        return 'HIGH'
    elif status == 'YELLOW':
        return 'MEDIUM'
    else:  # GREEN
        # Even GREEN assets with weak keys should be flagged
        if algorithm in ['RSA', 'DSA'] and key_size and key_size < 2048:
            return 'MEDIUM'
        return 'LOW'


# Quick test
status, buffer, explanation = mosca_classification(15, ORG_MIGRATION_YEARS, QDAY_YEARS_FROM_NOW)
print(f"\nMosca Test (15yr data lifetime, {ORG_MIGRATION_YEARS}yr migration, {QDAY_YEARS_FROM_NOW}yr to Q-Day):")
print(f"  Status: {status} | Buffer: {buffer:.1f} years | {explanation}")

## Step 4: Certificate and Key Extraction Functions

In [ ]:
def extract_cert_info(cert_path: str) -> Optional[Dict]:
    """
    Extract cryptographic metadata from an X.509 certificate.
    Returns a dict with algorithm, key_size, expiry, subject, etc.
    """
    try:
        with open(cert_path, 'rb') as f:
            data = f.read()
        
        # Try PEM format first, then DER
        try:
            cert = x509.load_pem_x509_certificate(data, default_backend())
        except Exception:
            cert = x509.load_der_x509_certificate(data, default_backend())
        
        pub_key = cert.public_key()
        algorithm, key_size = get_key_info(pub_key)
        
        # Extract subject CN
        try:
            subject = cert.subject.get_attributes_for_oid(
                x509.NameOID.COMMON_NAME
            )[0].value
        except (IndexError, Exception):
            subject = str(cert.subject)
        
        not_after = cert.not_valid_after_utc.date() if hasattr(
            cert, 'not_valid_after_utc'
        ) else cert.not_valid_after.date()
        
        days_remaining = (not_after - TODAY).days
        lifetime_years = max(days_remaining / 365.25, 0.1)
        
        return {
            'type': 'CERTIFICATE',
            'algorithm': algorithm,
            'key_size': key_size,
            'expiry': str(not_after),
            'days_remaining': days_remaining,
            'lifetime_years': round(lifetime_years, 2),
            'subject': subject[:60],
            'issuer': str(cert.issuer)[:60],
            'serial': str(cert.serial_number)[:20],
        }
    except Exception as e:
        return None


def extract_key_info(key_path: str) -> Optional[Dict]:
    """
    Extract metadata from a PEM-encoded private key file.
    """
    try:
        with open(key_path, 'rb') as f:
            data = f.read()
        
        # Try loading as private key (unencrypted)
        try:
            private_key = serialization.load_pem_private_key(
                data, password=None, backend=default_backend()
            )
        except Exception:
            # Try public key
            try:
                private_key = serialization.load_pem_public_key(
                    data, backend=default_backend()
                )
            except Exception:
                return None
        
        algorithm, key_size = get_key_info(private_key)
        
        return {
            'type': 'KEY',
            'algorithm': algorithm,
            'key_size': key_size,
            'expiry': 'N/A',
            'days_remaining': -1,  # Keys don't expire
            'lifetime_years': DEFAULT_LIFETIME_YEARS,
            'subject': 'Private Key',
            'issuer': 'N/A',
            'serial': 'N/A',
        }
    except Exception as e:
        return None


def get_key_info(key) -> Tuple[str, Optional[int]]:
    """
    Determine algorithm and key size from a cryptographic key object.
    """
    if isinstance(key, (RSAPrivateKey, RSAPublicKey)):
        return 'RSA', key.key_size
    elif isinstance(key, (EllipticCurvePrivateKey, EllipticCurvePublicKey)):
        curve_name = key.curve.name if hasattr(key, 'curve') else 'unknown'
        key_size = key.key_size if hasattr(key, 'key_size') else None
        return f'ECDSA ({curve_name})', key_size
    elif isinstance(key, (DSAPrivateKey, DSAPublicKey)):
        return 'DSA', key.key_size
    elif isinstance(key, ed25519.Ed25519PrivateKey):
        return 'Ed25519', 255
    elif isinstance(key, ed448.Ed448PrivateKey):
        return 'Ed448', 448
    else:
        return type(key).__name__, None


def is_quantum_vulnerable(algorithm: str, key_size: Optional[int]) -> bool:
    """
    Determine if an algorithm is vulnerable to Shor's algorithm.
    """
    vulnerable_algorithms = ['RSA', 'DSA', 'DH', 'ECDH']
    if any(alg in algorithm for alg in ['RSA', 'DSA', 'ECDSA', 'ECDH', 'Ed25519', 'Ed448']):
        return True
    return False


print("✅ Extraction functions loaded")

## Step 5: The Directory Scanner

In [ ]:
# File extensions to scan
CERT_EXTENSIONS = {'.crt', '.cer', '.pem', '.der', '.p7b', '.p7c'}
KEY_EXTENSIONS  = {'.key', '.pem', '.priv', '.private'}
BUNDLE_EXTENSIONS = {'.p12', '.pfx'}


def scan_directory(scan_path: str) -> List[Dict]:
    """
    Walk a directory tree and extract info from all cryptographic files found.
    Returns a list of asset dictionaries with Mosca classification.
    """
    results = []
    errors = []
    scan_root = Path(scan_path)
    
    if not scan_root.exists():
        print(f"⚠️  Directory not found: {scan_path}")
        return results
    
    print(f"🔍 Scanning: {scan_path}")
    files_checked = 0
    
    for file_path in scan_root.rglob('*'):
        if not file_path.is_file():
            continue
        
        ext = file_path.suffix.lower()
        info = None
        
        if ext in CERT_EXTENSIONS:
            # Try as certificate first
            info = extract_cert_info(str(file_path))
            if info is None:
                # Might be a key in a .pem file
                info = extract_key_info(str(file_path))
        elif ext in KEY_EXTENSIONS:
            info = extract_key_info(str(file_path))
            if info is None:
                info = extract_cert_info(str(file_path))
        
        files_checked += 1
        
        if info is not None:
            # Apply Mosca Inequality
            status, buffer, explanation = mosca_classification(
                info['lifetime_years'],
                ORG_MIGRATION_YEARS,
                QDAY_YEARS_FROM_NOW
            )
            
            quantum_vulnerable = is_quantum_vulnerable(
                info['algorithm'], info.get('key_size')
            )
            
            priority = get_priority(status, info.get('key_size'), info['algorithm'])
            
            result = {
                'file': str(file_path.relative_to(scan_root)),
                'type': info['type'],
                'algorithm': info['algorithm'],
                'key_size': info.get('key_size', 'N/A'),
                'expiry': info['expiry'],
                'days_remaining': info['days_remaining'],
                'lifetime_years': info['lifetime_years'],
                'subject': info['subject'],
                'quantum_vulnerable': quantum_vulnerable,
                'mosca_status': status,
                'mosca_buffer_years': round(buffer, 2),
                'mosca_explanation': explanation,
                'priority': priority,
            }
            results.append(result)
    
    print(f"   Files checked: {files_checked}")
    print(f"   Crypto assets found: {len(results)}")
    
    return results


def generate_csv_report(results: List[Dict], output_path: str) -> None:
    """
    Write the prioritized migration report to a CSV file.
    Sorted by priority: CRITICAL > HIGH > MEDIUM > LOW.
    """
    priority_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
    sorted_results = sorted(results, key=lambda x: priority_order.get(x['priority'], 99))
    
    if not sorted_results:
        print("No results to write.")
        return
    
    with open(output_path, 'w', newline='') as csvfile:
        fieldnames = list(sorted_results[0].keys())
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(sorted_results)
    
    print(f"\n📄 Report written to: {output_path}")


print("✅ Scanner functions loaded")

## Step 6: Generate Sample Certificates for Testing

This creates a realistic test environment with certificates of varying algorithms, key sizes, and expiration dates — representing the kind of environment a real organization might have.

In [ ]:
import ipaddress
from cryptography.hazmat.primitives.asymmetric import rsa as rsa_module
from cryptography.hazmat.primitives.asymmetric import ec as ec_module
from cryptography.hazmat.primitives import hashes
from cryptography.x509.oid import NameOID

def create_test_certificate(
    cn: str,
    algorithm: str = 'RSA',
    key_size: int = 2048,
    valid_days: int = 365
) -> Tuple[bytes, bytes]:
    """
    Create a self-signed test certificate.
    Returns (cert_pem, key_pem).
    """
    if algorithm == 'RSA':
        private_key = rsa_module.generate_private_key(
            public_exponent=65537,
            key_size=key_size,
            backend=default_backend()
        )
        hash_alg = hashes.SHA256()
    elif algorithm == 'ECDSA':
        private_key = ec_module.generate_private_key(
            ec_module.SECP256R1() if key_size <= 256 else ec_module.SECP384R1(),
            backend=default_backend()
        )
        hash_alg = hashes.SHA256()
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    
    subject = issuer = x509.Name([
        x509.NameAttribute(NameOID.COUNTRY_NAME, "US"),
        x509.NameAttribute(NameOID.ORGANIZATION_NAME, "Test Organization"),
        x509.NameAttribute(NameOID.COMMON_NAME, cn),
    ])
    
    now = datetime.datetime.utcnow()
    cert = (
        x509.CertificateBuilder()
        .subject_name(subject)
        .issuer_name(issuer)
        .public_key(private_key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(now)
        .not_valid_after(now + datetime.timedelta(days=valid_days))
        .add_extension(x509.SubjectAlternativeName([x509.DNSName(cn)]), critical=False)
        .sign(private_key, hash_alg, default_backend())
    )
    
    cert_pem = cert.public_bytes(serialization.Encoding.PEM)
    key_pem = private_key.private_bytes(
        serialization.Encoding.PEM,
        serialization.PrivateFormat.TraditionalOpenSSL,
        serialization.NoEncryption()
    )
    
    return cert_pem, key_pem


# Create test directory structure
TEST_DIR = tempfile.mkdtemp(prefix='pqc_scan_')

test_certs = [
    # (filename_prefix, cn, algorithm, key_size, valid_days)
    ('web-server', 'web.example.com', 'RSA', 2048, 365),        # ~1yr, RSA-2048
    ('legacy-api', 'api-legacy.example.com', 'RSA', 1024, 730), # ~2yr, WEAK RSA-1024
    ('payment-gw', 'payments.example.com', 'RSA', 4096, 3650),  # ~10yr, strong RSA
    ('client-auth', 'client.example.com', 'ECDSA', 256, 180),   # ~0.5yr, ECDSA
    ('long-lived', 'root.example.com', 'RSA', 2048, 5475),      # ~15yr, RSA root
    ('modern-tls', 'api.example.com', 'ECDSA', 384, 365),       # ~1yr, P-384
]

print("📁 Creating test certificate environment...")
for prefix, cn, algo, size, days in test_certs:
    cert_pem, key_pem = create_test_certificate(cn, algo, size, days)
    cert_path = os.path.join(TEST_DIR, f'{prefix}.crt')
    key_path  = os.path.join(TEST_DIR, f'{prefix}.key')
    with open(cert_path, 'wb') as f: f.write(cert_pem)
    with open(key_path,  'wb') as f: f.write(key_pem)
    print(f"   ✅ Created {prefix}.crt ({algo}-{size}, {days} days)")

print(f"\n📂 Test directory: {TEST_DIR}")

## Step 7: Run the PQC Migration Scanner

In [ ]:
# Run the scanner on our test directory
print("=" * 60)
print("PQC MIGRATION SCANNER — RUNNING")
print(f"Q-Day Estimate: {TODAY.year + QDAY_YEARS_FROM_NOW} ({QDAY_YEARS_FROM_NOW} years)")
print(f"Migration Time: {ORG_MIGRATION_YEARS} years")
print("=" * 60)

scan_results = scan_directory(TEST_DIR)

# Generate CSV report
REPORT_PATH = '/tmp/pqc_migration_report.csv'
generate_csv_report(scan_results, REPORT_PATH)

print("\n✅ Scan complete")

## Step 8: Display Results and Summary

In [ ]:
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

if HAS_PANDAS and scan_results:
    df = pd.DataFrame(scan_results)
    
    # Display prioritized table
    priority_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
    df['priority_rank'] = df['priority'].map(priority_order)
    df_sorted = df.sort_values('priority_rank').drop(columns=['priority_rank'])
    
    display_cols = ['file', 'algorithm', 'key_size', 'expiry', 
                    'lifetime_years', 'mosca_status', 'priority']
    
    # Color-code by status
    def color_status(val):
        colors = {'RED': 'background-color: #ffcccc', 
                  'YELLOW': 'background-color: #ffffcc',
                  'GREEN': 'background-color: #ccffcc'}
        return colors.get(val, '')
    
    styled = df_sorted[display_cols].style.applymap(
        color_status, subset=['mosca_status']
    )
    
    print("\n📊 PQC MIGRATION REPORT — PRIORITIZED")
    print("=" * 60)
    display(styled)
    
    # Summary statistics
    print("\n📈 SUMMARY")
    print("-" * 40)
    status_counts = df['mosca_status'].value_counts()
    priority_counts = df['priority'].value_counts()
    
    for status, color in [('RED', '🔴'), ('YELLOW', '🟡'), ('GREEN', '🟢')]:
        count = status_counts.get(status, 0)
        print(f"  {color} {status}: {count} asset(s)")
    
    print()
    for priority in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']:
        count = priority_counts.get(priority, 0)
        print(f"  Priority {priority}: {count} asset(s)")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Mosca Status Pie
    status_data = [status_counts.get(s, 0) for s in ['RED', 'YELLOW', 'GREEN']]
    axes[0].pie(status_data, labels=['RED', 'YELLOW', 'GREEN'], 
                colors=['#ff6b6b', '#ffd93d', '#6bcb77'],
                autopct='%1.0f%%', startangle=90)
    axes[0].set_title('Mosca Inequality\nRisk Classification')
    
    # Priority Bar Chart
    priorities = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
    p_counts = [priority_counts.get(p, 0) for p in priorities]
    colors = ['#d62728', '#ff7f0e', '#ffdd57', '#2ca02c']
    axes[1].bar(priorities, p_counts, color=colors)
    axes[1].set_title('Migration Priority\nDistribution')
    axes[1].set_ylabel('Asset Count')
    
    plt.tight_layout()
    plt.savefig('/tmp/pqc_scan_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n📊 Chart saved to /tmp/pqc_scan_summary.png")

else:
    # Text-only display
    print("\n📊 PQC MIGRATION REPORT — PRIORITIZED")
    print("-" * 80)
    priority_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
    sorted_results_display = sorted(scan_results, 
                                     key=lambda x: priority_order.get(x['priority'], 99))
    for r in sorted_results_display:
        icon = {'RED': '🔴', 'YELLOW': '🟡', 'GREEN': '🟢'}.get(r['mosca_status'], '⚪')
        print(f"  {icon} [{r['priority']:8s}] {r['file'][:40]:40s} | "
              f"{r['algorithm']:15s} {str(r['key_size']):6s} | "
              f"Expires: {r['expiry']}")
        print(f"           Mosca: {r['mosca_explanation']}")

## Step 9: Scan Your Own System (Optional)

To scan a real directory, change the path below. Common locations:
- **Linux/Mac:** `/etc/ssl/`, `/etc/pki/`, `~/.ssl/`
- **Windows:** (Export certificates from certmgr.msc to a folder first)
- **Docker containers:** Mount the host cert directory and scan it

⚠️ **Security Note:** This scanner only reads certificate metadata — it does not extract or transmit private key material.

In [ ]:
# Change this to scan your own certificate directory
# CUSTOM_SCAN_PATH = '/etc/ssl'  
# CUSTOM_SCAN_PATH = '/path/to/your/certs'

CUSTOM_SCAN_PATH = None  # Set to a path to enable

if CUSTOM_SCAN_PATH:
    print(f"🔍 Scanning: {CUSTOM_SCAN_PATH}")
    custom_results = scan_directory(CUSTOM_SCAN_PATH)
    
    CUSTOM_REPORT = '/tmp/pqc_custom_report.csv'
    generate_csv_report(custom_results, CUSTOM_REPORT)
    
    print(f"\n📄 Full report saved to: {CUSTOM_REPORT}")
    print("Download this file for your deliverable.")
else:
    print("💡 Set CUSTOM_SCAN_PATH to scan your own certificate directory.")
    print(f"\n📄 Sample report is available at: {REPORT_PATH}")
    
    # Download the sample report
    try:
        from google.colab import files
        files.download(REPORT_PATH)
        print("✅ CSV report downloaded")
    except ImportError:
        print("(Not in Colab — report is at /tmp/pqc_migration_report.csv)")

## Deliverable Instructions

### What to Submit

1. **Working script:** This completed notebook, with your scan results populated

2. **Sample output:** The CSV report from your scan, with at least 5 entries classified across GREEN/YELLOW/RED. You may use the sample certificates generated in Step 6, or scan your own system.

3. **500-word executive summary** — Write it in the cell below, answering:
   - What did you find? (How many assets, what algorithms, what key sizes?)
   - What is the priority order for migration? (Start with CRITICAL and RED)
   - What is the headline Mosca risk exposure? (How many assets are in deficit?)
   - What is your top recommendation for the CISO?

### Grading Criteria

| Component | Points |
|-----------|--------|
| Working scanner that produces output | 30 |
| CSV report with ≥5 entries across GREEN/YELLOW/RED | 25 |
| Executive summary: accurate Mosca analysis | 25 |
| Executive summary: actionable CISO recommendation | 20 |
| **Total** | **100** |

## Your Executive Summary

*Write your 500-word executive summary here. Replace this text with your analysis.*

---

**TO:** Chief Information Security Officer  
**FROM:** [Your Name]  
**RE:** PQC Migration Scanner — Initial Findings  
**DATE:** [Date]

**Summary of Findings:**

[Your analysis here — minimum 500 words]

**Priority Order for Migration:**

1. [CRITICAL items first]
2. [HIGH items next]
3. ...

**Mosca Risk Exposure:**

[How many assets are in RED? What does that mean in business terms?]

**Top Recommendation:**

[Your single most important recommendation for the CISO]